# Feature Creation Pipeline
This notebook contains all steps for creating item features from raw data: category parsing, brand cleaning, text cleaning, feature extraction from title/description/feature, numeric parsing, feature cleaning, and SBERT embeddings.

## **Import the packages and get the data**

In [1]:
# Import packages
import pandas as pd
import numpy as np
import re
import io
import matplotlib.pyplot as plt
import seaborn as sns
import math
from scipy.sparse import csr_matrix
import nltk
nltk.download('wordnet', quiet=True)
from nltk.stem import WordNetLemmatizer
#
from contextlib import redirect_stdout

In [2]:
# Set display option to show full content of columns
pd.options.display.max_colwidth = 50

# Show all the columns
pd.options.display.max_columns = None

# Turn off scientific notation for pandas DataFrames
pd.options.display.float_format = '{:.2f}'.format

In [3]:
# Take the dataframes
# file_reviews = '/content/drive/MyDrive/Recommendation Engine BP/Home_and_Kitchen_filtered.csv'
file_reviews = '../data/Home_and_Kitchen_filtered.csv'
df_reviews = pd.read_csv(file_reviews).drop_duplicates()

file_items = '../data/meta_Home_and_Kitchen_filtered.csv'
df_items = pd.read_csv(file_items).drop_duplicates()

print(f"df_reviews: {len(df_reviews):,} rows, {df_reviews['asin'].nunique():,} unique ASINs")
print(f"df_items: {len(df_items):,} rows, {df_items['asin'].nunique():,} unique ASINs")

/var/folders/86/_khp3pb10vg5vtbr6fmndrxc0000gn/T/ipykernel_45271/752698439.py:4: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_reviews = pd.read_csv(file_reviews).drop_duplicates()
/var/folders/86/_khp3pb10vg5vtbr6fmndrxc0000gn/T/ipykernel_45271/752698439.py:7: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_items = pd.read_csv(file_items).drop_duplicates()


df_reviews: 6,667,845 rows, 189,172 unique ASINs
df_items: 1,285,392 rows, 1,285,392 unique ASINs


In [4]:
# Merge reviews with items using left_join
df_combined = df_reviews.merge(df_items, left_on='asin', right_on='asin', how = 'left')

# Convert 'unixReviewTime' to datetime format
df_combined['unixReviewTime'] = pd.to_datetime(df_combined['unixReviewTime'], unit='s')

## Check **"Date"** Variable

In [5]:
# Compare earliest review date with the date variable of products
df_earliest_review = (
    df_combined.groupby('asin')
    .agg(earliest_review_date=('unixReviewTime', 'min'), date=('date', 'first'))
    .reset_index()
)
df_earliest_review

,asin,earliest_review_date,date
0,0560467893,2015-05-07,None
1,0681795107,2010-11-26,"August 1, 2006"
2,0768205921,2012-02-15,None
3,0805469613,2011-05-16,"October 31, 2006"
4,0983124248,2013-01-27,None
...,...,...,...
189167,B01HJCREGO,2016-09-01,None
189168,B01HJEJDBQ,2016-08-08,None
189169,B01HJEKGHQ,2016-11-07,None
189170,B01HJEOT2E,2016-11-11,"June 25, 2016"


In [6]:
# Percentage of products with missing date
pct_missing = df_earliest_review['date'].isna().mean() * 100
print(f"Percentage of products with missing date: {pct_missing:.2f}%")

# Compare date with earliest_review_date
df_earliest_review['date_parsed'] = pd.to_datetime(df_earliest_review['date'], format='mixed', errors='coerce')
df_earliest_review['date_before_review'] = (df_earliest_review['date_parsed'] <= df_earliest_review['earliest_review_date']).astype(int)
df_earliest_review[['asin', 'earliest_review_date', 'date', 'date_before_review']]

Percentage of products with missing date: 51.50%


,asin,earliest_review_date,date,date_before_review
0,0560467893,2015-05-07,None,0
1,0681795107,2010-11-26,"August 1, 2006",1
2,0768205921,2012-02-15,None,0
3,0805469613,2011-05-16,"October 31, 2006",1
4,0983124248,2013-01-27,None,0
...,...,...,...,...
189167,B01HJCREGO,2016-09-01,None,0
189168,B01HJEJDBQ,2016-08-08,None,0
189169,B01HJEKGHQ,2016-11-07,None,0
189170,B01HJEOT2E,2016-11-11,"June 25, 2016",1


**Conclusion:** For most of the items, the date is before the first review date meaning that the date is probably that of when the product was first listed.

## **Item Features**

**Variables to consider:** <br> category: divided into 6 variables (can be used) <br> title <br> brand (needs discussion) <br> price (can be used) <br> description <br> feature <br> style: comes from reviews (needs discussion) <br> main_cat (Mostly "Amazon Home", cannot be used) <br> rank <br> date (can be used) <br> image <br> tech1/tech2

**Category**

In [7]:
import ast

# Parse category strings into lists and count levels per product
category_levels = (
   df_items.dropna(subset=['category'])['category']
   .apply(lambda x: len(ast.literal_eval(x)))
)

print("Category level distribution:")
print(category_levels.value_counts().sort_index())

Category level distribution:
category
2      41751
3     214066
4     536324
5     360842
6     117173
7       7986
8       1143
9       1756
10      2018
11      1803
12       405
13       110
14        12
16         3
Name: count, dtype: int64


In [8]:
# Create variables from the 6 categories
cat_lists = df_items['category'].dropna().apply(ast.literal_eval)
for i in range(6):
    df_items[f'cat_{i+1}'] = cat_lists.apply(lambda x: x[i] if len(x) > i else None)

In [9]:
df_items[['asin', 'cat_1', 'cat_2', 'cat_3', 'cat_4', 'cat_5', 'cat_6']]

,asin,cat_1,cat_2,cat_3,cat_4,cat_5,cat_6
0,0001487795,Home & Kitchen,Kitchen & Dining,Dining & Entertaining,Dinnerware,Plates,Dinner Plates
1,0002020300,Home & Kitchen,Home Dcor,Candles & Holders,Candles,None,None
2,0006564224,Home & Kitchen,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,Wine & Champagne Glasses,None
3,0009046461,Home & Kitchen,Bath,Bathroom Accessories,None,None,None
4,0234937912,Home & Kitchen,Home Dcor,Home Fragrance,Incense & Incense Holders,Incense,None
...,...,...,...,...,...,...,...
1300535,B01HJHTC6O,Home & Kitchen,Kitchen & Dining,Small Appliance Parts & Accessories,None,None,None
1300536,B01HJH0J4S,Home & Kitchen,Kitchen & Dining,Small Appliances,None,None,None
1300537,B01HJGJNWS,Home & Kitchen,Kitchen & Dining,Cookware,Roasting Pans,None,None
1300538,B01HJHOITU,Home & Kitchen,Home Dcor,Artificial Plants & Flowers,Artificial Flowers,None,None


**Conclusion**: category consists of several levels of descriptions. After checking the number of items with different number of categories, we found that 98.8% of items have <= 6 categories. Thus, we created 6 columns for the categories naming cat_1 ... cat_6.

**brand**

In [10]:
df_items['brand'].head(10)

0        Waechtersbach USA
1                    Vicks
2      Artistic Churchware
3                   Mysore
4                Patanjali
5    Get Motivated Posters
6                    Cello
7               Scholastic
8           My Little Pony
9                 Pfeiffer
Name: brand, dtype: object

In [11]:
# Clean brand: lowercase and remove special characters and spaces
df_items['brand_clean'] = (
    df_items['brand']
    .str.lower()
    .str.replace(r'[^a-z0-9]', '', regex=True)
)
df_items[['asin', 'brand', 'brand_clean']].head(10)

,asin,brand,brand_clean
0,0001487795,Waechtersbach USA,waechtersbachusa
1,0002020300,Vicks,vicks
2,0006564224,Artistic Churchware,artisticchurchware
3,0009046461,Mysore,mysore
4,0234937912,Patanjali,patanjali
5,0250459655,Get Motivated Posters,getmotivatedposters
6,0326591516,Cello,cello
7,0439903491,Scholastic,scholastic
8,0456680012,My Little Pony,mylittlepony
9,0470902884,Pfeiffer,pfeiffer


**Conclusion:** We cleaned the brand variable by removing the stopwords and converting to lowercase.

**main_cat:**

In [12]:
df_items['main_cat'].value_counts()

main_cat
Amazon Home                                                                                                                                                                 1143571
Tools & Home Improvement                                                                                                                                                      21306
Toys & Games                                                                                                                                                                  16325
Health & Personal Care                                                                                                                                                        12414
<img src="https://images-na.ssl-images-amazon.com/images/G/01/nav2/images/gui/amazon-fashion-store-new._CB520838675_.png" class="nav-categ-image" alt="AMAZON FASHION"/>      12395
Arts, Crafts & Sewing                                                                      

**Conclusion** main_cat variable is mostly "Amazon Home". Some other values are also available but cannot be used for feature extractions

## Text Cleaning: title, description, feature
Remove stopwords and lemmatize to create `title_1`, `description_1`, `feature_1`.

In [13]:
# Upload the stopwords and lemmatizer
import ssl

# Fix SSL certificate issue on macOS/Anaconda
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

lemmatizer = WordNetLemmatizer()

nltk_stop_words = set(stopwords.words('english'))

In [14]:
# Update the list of stopwords to keep the necessary ones
def parse_list_string(text):
    """Convert string representation of a list into a single joined string.
    e.g. "['item1', 'item2']" -> 'item1 item2'
    Returns empty string for NaN or empty lists.
    """
    if pd.isna(text) or text == '[]':
        return ''
    try:
        items = ast.literal_eval(text)
        if isinstance(items, list):
            return ' '.join(str(item) for item in items)
    except (ValueError, SyntaxError):
        pass
    return str(text)


# Words to keep even though they are NLTK stopwords (useful for product features)
stopwords_to_keep = {'no', 'not', 'non', 'off', 'self'}
custom_stop_words = set(nltk_stop_words) - stopwords_to_keep

In [15]:
# Map the number words to digits (1-20)
# Create the clean_text function
number_words = {
    'one': '1', 'two': '2', 'three': '3', 'four': '4', 'five': '5',
    'six': '6', 'seven': '7', 'eight': '8', 'nine': '9', 'ten': '10',
    'eleven': '11', 'twelve': '12', 'thirteen': '13', 'fourteen': '14',
    'fifteen': '15', 'sixteen': '16', 'seventeen': '17', 'eighteen': '18',
    'nineteen': '19', 'twenty': '20'
}
number_pattern = re.compile(r'\b(' + '|'.join(number_words.keys()) + r')\b')

def clean_text(text):
    """Lowercase, convert number words to digits, remove special characters
    (keep letters, numbers, hyphens, decimals), remove stopwords, and lemmatize."""
    if pd.isna(text) or text == '':
        return ''

    text = text.lower()

    # 1. Convert number words to digits (e.g., "five-drawer" -> "5-drawer")
    text = number_pattern.sub(lambda m: number_words[m.group()], text)

    # 2. Standardize unit hyphens (e.g., "12-pc" -> "12 pc")
    text = re.sub(r'(\d+)-(pc|inch|oz|lb|piece)', r'\1 \2', text)

    # 3. Keep alphanumeric, spaces, decimals, and hyphens
    text = re.sub(r'[^a-z0-9\s.\-]', ' ', text)

    # 4. Keep dots between digits (10.5), remove others
    text = re.sub(r'(?<!\d)\.(?!\d)', ' ', text)

    # 5. Collapse spaces
    text = ' '.join(text.split())

    # 6. Tokenize, remove stopwords, and lemmatize
    tokens = [lemmatizer.lemmatize(w) for w in text.split() if w not in custom_stop_words]

    return ' '.join(tokens)

## Extract features from product **title**

In [16]:
import json

# Load global filters (expressions to remove from titles)
with open('../data/global_filters.json') as f:
    global_filters = json.load(f)

# Sort by length descending so multi-word expressions are matched first
global_filters_sorted = sorted(global_filters, key=len, reverse=True)

# Build regex pattern from global filters (escaped, word-boundary matched)
filter_pattern = '|'.join(r'\b' + re.escape(expr) + r'\b' for expr in global_filters_sorted)

def remove_global_filters(text):
    """Remove global filter expressions from text."""
    if not text:
        return text
    return re.sub(filter_pattern, '', text).strip()

# Apply clean_text then remove global filter expressions
df_items['title_cleaned'] = (
    df_items['title']
    .fillna('')
    .apply(clean_text)
    .apply(remove_global_filters)
)

# Collapse any double spaces left after filter removal
df_items['title_cleaned'] = df_items['title_cleaned'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Show sample results
print(f"Total items: {len(df_items)}")
print(f"Non-empty title_cleaned: {(df_items['title_cleaned'] != '').sum()}")
print(f"\nSample before/after:")
sample = df_items[df_items['title_cleaned'] != ''][['title', 'title_cleaned']].head(10)
for _, row in sample.iterrows():
    print(f"  BEFORE: {row['title'][:80]}")
    print(f"  AFTER:  {row['title_cleaned'][:80]}")
    print()

Total items: 1285392
Non-empty title_cleaned: 1285350

Sample before/after:
  BEFORE: You Are Special Today Red Plate [With Red Pen]
  AFTER:  special today red plate red pen

  BEFORE: Vicks Inhaler Relief for Cold Sinus Nasal Congestion Allergy
  AFTER:  vicks inhaler relief cold sinus nasal congestion allergy

  BEFORE: Artistic Churchware Communion Cup Filler: RW525
  AFTER:  artistic churchware communion cup filler rw525

  BEFORE: 4 BARS! Mysore Sandal Soap 70grams FAST SHIPPING
  AFTER:  4 bar mysore sandal soap 70grams fast shipping

  BEFORE: AROGYA VATI (40gm) by popeye seller
  AFTER:  arogya vati 40gm popeye seller

  BEFORE: Nikola Tesla Photo&hellip;Nikola Tesla Quotes Poster Print (12 inch X 18 inch, R
  AFTER:  nikola tesla photo nikola tesla quote poster print 12 inch x 18 inch rolled

  BEFORE: Set of 15 Cello Butterflow Blue, Red &amp; Black Ball Pen - Brand New - India
  AFTER:  set 15 cello butterflow blue red black ball pen - brand - india

  BEFORE: Scholastic Pi

In [17]:
# Get master_metadata.json
with open('../data/master_metadata.json') as f:
    master_metadata = json.load(f)

# Keep only items whose cat_3 is in master_metadata
valid_cat3 = set(master_metadata.keys())
df_items_filtered = df_items[df_items['cat_3'].isin(valid_cat3)].copy()

print(f"Items with cat_3 in master_metadata: {len(df_items_filtered):,} / {len(df_items):,}")
print(f"Categories matched: {df_items_filtered['cat_3'].nunique()} / {len(valid_cat3)}")
print(f"\nBreakdown by cat_2:")
print(df_items_filtered['cat_2'].value_counts())

Items with cat_3 in master_metadata: 1,134,566 / 1,285,392
Categories matched: 69 / 69

Breakdown by cat_2:
cat_2
Kitchen & Dining    429949
Home Dcor           343481
Bedding             130115
Wall Art            108969
Furniture            69232
Bath                 51945
Kids' Home Store       875
Name: count, dtype: int64


In [18]:
# Create the extract_features function. The function is used to extract the features from title and store as new variable: extracted_features
def extract_features(title, cat_3):
    """Extract features from title using master_matadata.json for given cat_3 variable"""
    if pd.isna(cat_3) or cat_3 not in master_metadata or pd.isna(title) or title == '':
        return {}

    features = {}
    title_lower = title.lower()

    for field, spec in master_metadata[cat_3].items():
        if spec['type'] == 'dictionary':
            for val in sorted(spec['values'], key=len, reverse=True):
                if re.search(r'\b' + re.escape(val) + r'\b', title_lower):
                    features[field] = val
                    break
        elif spec['type'] == 'regex':
            for pattern in spec['patterns']:
                match = re.search(pattern, title_lower)
                if match:
                    result = match.group(0)
                    # For Dimensions, capture any trailing unit not already in the match
                    if field == 'Dimensions':
                        remaining = title_lower[match.end():]
                        unit_match = re.match(
                            r'[\s-]*(inch|inchs|in|cm|mm|ft|foot|feet|meter|m|quot|"|\"|\')',
                            remaining
                        )
                        if unit_match:
                            result = title_lower[match.start():match.end() + unit_match.end()]
                    features[field] = result
                    break
    return features

In [19]:
df_items_filtered['extracted_features_title'] = df_items_filtered.apply(
    lambda row: extract_features(row['title_cleaned'], row.get('cat_3')), axis=1
)

# Summary
total = len(df_items_filtered)
with_features = (df_items_filtered['extracted_features_title'].apply(len) > 0).sum()
print(f"Products with extracted features (title): {with_features:,} / {total:,} ({with_features/total*100:.1f}%)")

from collections import Counter
field_counts = Counter()
for feat_dict in df_items_filtered['extracted_features_title']:
    for key in feat_dict:
        field_counts[key] += 1

print(f"\nFeature coverage:")
for field, count in field_counts.most_common():
    print(f"  {field}: {count:,} ({count/total*100:.1f}%)")

# Sample
sample = df_items_filtered[df_items_filtered['extracted_features_title'].apply(len) > 0][['title_cleaned', 'cat_3', 'extracted_features_title']].head(5)
for _, row in sample.iterrows():
    print(f"\n  cat_3: {row['cat_3']} | title: {row['title_cleaned'][:70]}")
    print(f"  features: {row['extracted_features_title']}")

Products with extracted features (title): 1,048,500 / 1,134,566 (92.4%)

Feature coverage:
  Product_Type: 724,822 (63.9%)
  Material: 389,648 (34.3%)
  Color: 351,924 (31.0%)
  Dimensions: 302,258 (26.6%)
  Features: 300,370 (26.5%)
  Piece_Count: 188,953 (16.7%)
  Brand: 144,362 (12.7%)
  Theme: 117,780 (10.4%)
  Size: 76,125 (6.7%)
  Capacity_Volume: 71,510 (6.3%)
  Shape: 45,990 (4.1%)
  Sub_Type: 29,245 (2.6%)
  Shape_Style: 23,052 (2.0%)
  Thread_Count: 13,897 (1.2%)
  Scent: 8,155 (0.7%)
  Pocket_Depth: 2,852 (0.3%)
  Capacity: 2,304 (0.2%)
  Power_Rating: 2,033 (0.2%)
  Capacity_Cups: 1,833 (0.2%)
  Filter_Rating: 694 (0.1%)
  Voltage: 595 (0.1%)
  Weight: 200 (0.0%)
  Stage_Count: 181 (0.0%)
  Part_Number: 131 (0.0%)
  Density_Weight: 107 (0.0%)
  Bar_Pressure: 47 (0.0%)

  cat_3: Dining & Entertaining | title: special today red plate red pen
  features: {'Color': 'red'}

  cat_3: Dining & Entertaining | title: artistic churchware communion cup filler rw525
  features: {'Produ

In [20]:
# Unit map for standardizing unit names (applied after feature parsing)
unit_map = {
    'feet': 'ft', 'foot': 'ft',
    'ounce': 'oz',
    'piece': 'pc', 'pieces': 'pc', 'pcs': 'pc',
    'quart': 'qt',
    'liter': 'l',
    'inch': 'in', 'inchs': 'in', 'inch deep': 'in',
    'pound': 'lb',
    'gallon': 'gal',
    'gram': 'g', 'gm': 'g',
    'centimeter': 'cm',
    'millimeter': 'mm',
    'watt': 'w',
    'volt': 'v',
    'dozen': 'dz',
    'cu ft': 'cubic foot', 'cuft': 'cubic foot',
    'tc': 'thread count', 'thread': 'thread count', 'threadcount': 'thread count',
    'pk': 'pocket',
    'count': 'count', 'ct': 'count',
    'pillowcase': 'pillow case',
}

## Extract features from product **description**

In [21]:
# Clean description: parse list string, clean text, remove global filters
df_items_filtered['description_cleaned'] = (
    df_items_filtered['description']
    .fillna('')
    .apply(parse_list_string)
    .apply(clean_text)
    .apply(remove_global_filters)
)

# Collapse any double spaces left after filter removal
df_items_filtered['description_cleaned'] = df_items_filtered['description_cleaned'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Summary
non_empty = (df_items_filtered['description_cleaned'] != '').sum()
print(f"Non-empty description_cleaned: {non_empty:,} / {len(df_items_filtered):,}")

# Sample
sample = df_items_filtered[df_items_filtered['description_cleaned'] != ''][['description', 'description_cleaned']].head(3)
for _, row in sample.iterrows():
    print(f"\n  BEFORE: {str(row['description'])[:100]}")
    print(f"  AFTER:  {row['description_cleaned'][:100]}")

Non-empty description_cleaned: 1,010,980 / 1,134,566

  BEFORE: ['It was a time honored tradition among the early American families that when someone deserved speci
  AFTER:  time honored tradition among early american family someone deserved special praise attention served 

  BEFORE: ['VICKS INHALER relieves stuffy noses helps sinus congestion breathe easy great for allergy season m
  AFTER:  vicks inhaler relief stuffy nose help sinus congestion breathe great allergy season made india

  BEFORE: ['16 oz squeeze bottle, 1 lb.']
  AFTER:  16 oz squeeze bottle 1 lb


In [22]:
# Extract features from description
df_items_filtered['extracted_features_description'] = df_items_filtered.apply(
    lambda row: extract_features(row['description_cleaned'], row.get('cat_3')), axis=1
)

# Summary
total = len(df_items_filtered)
with_features = (df_items_filtered['extracted_features_description'].apply(len) > 0).sum()
print(f"Products with extracted features (description): {with_features:,} / {total:,} ({with_features/total*100:.1f}%)")

field_counts_desc = Counter()
for feat_dict in df_items_filtered['extracted_features_description']:
    for key in feat_dict:
        field_counts_desc[key] += 1

print(f"\nFeature coverage (description):")
for field, count in field_counts_desc.most_common():
    print(f"  {field}: {count:,} ({count/total*100:.1f}%)")

Products with extracted features (description): 915,857 / 1,134,566 (80.7%)

Feature coverage (description):
  Product_Type: 581,288 (51.2%)
  Material: 536,098 (47.3%)
  Features: 430,530 (37.9%)
  Color: 282,485 (24.9%)
  Dimensions: 275,823 (24.3%)
  Theme: 117,355 (10.3%)
  Piece_Count: 110,004 (9.7%)
  Brand: 78,141 (6.9%)
  Capacity_Volume: 67,225 (5.9%)
  Shape: 38,086 (3.4%)
  Size: 35,713 (3.1%)
  Sub_Type: 34,157 (3.0%)
  Shape_Style: 15,930 (1.4%)
  Scent: 9,902 (0.9%)
  Thread_Count: 8,923 (0.8%)
  Power_Rating: 3,767 (0.3%)
  Pocket_Depth: 3,279 (0.3%)
  Capacity: 2,484 (0.2%)
  Capacity_Cups: 1,629 (0.1%)
  Filter_Rating: 1,406 (0.1%)
  Voltage: 1,292 (0.1%)
  Part_Number: 384 (0.0%)
  Bar_Pressure: 280 (0.0%)
  Stage_Count: 206 (0.0%)
  Density_Weight: 204 (0.0%)
  Weight: 94 (0.0%)


## Extract features from product **feature** (bullet points)

In [23]:
# Clean feature: parse list string, clean text, remove global filters
df_items_filtered['feature_cleaned'] = (
    df_items_filtered['feature']
    .fillna('')
    .apply(parse_list_string)
    .apply(clean_text)
    .apply(remove_global_filters)
)

# Collapse any double spaces left after filter removal
df_items_filtered['feature_cleaned'] = df_items_filtered['feature_cleaned'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Summary
non_empty = (df_items_filtered['feature_cleaned'] != '').sum()
print(f"Non-empty feature_cleaned: {non_empty:,} / {len(df_items_filtered):,}")

# Sample
sample = df_items_filtered[df_items_filtered['feature_cleaned'] != ''][['feature', 'feature_cleaned']].head(3)
for _, row in sample.iterrows():
    print(f"\n  BEFORE: {str(row['feature'])[:100]}")
    print(f"  AFTER:  {row['feature_cleaned'][:100]}")

Non-empty feature_cleaned: 967,588 / 1,134,566

  BEFORE: ['Religious Supply Center', 'RW-525', 'Communion Cup Filler']
  AFTER:  religious supply center rw-525 communion cup filler

  BEFORE: ['Set of 15 Blue, Red & Black Ball Pen']
  AFTER:  set 15 blue red black ball pen

  BEFORE: ['Software: Play I SPY games to uncover the hidden treasure of three mysterious pirates!', 'Hard cov
  AFTER:  software play spy game uncover hidden treasure 3 mysterious pirate hard cover book read book inspire


In [24]:
# Extract features from feature (bullet points)
df_items_filtered['extracted_features_feature'] = df_items_filtered.apply(
    lambda row: extract_features(row['feature_cleaned'], row.get('cat_3')), axis=1
)

# Summary
total = len(df_items_filtered)
with_features = (df_items_filtered['extracted_features_feature'].apply(len) > 0).sum()
print(f"Products with extracted features (feature): {with_features:,} / {total:,} ({with_features/total*100:.1f}%)")

field_counts_feat = Counter()
for feat_dict in df_items_filtered['extracted_features_feature']:
    for key in feat_dict:
        field_counts_feat[key] += 1

print(f"\nFeature coverage (feature column):")
for field, count in field_counts_feat.most_common():
    print(f"  {field}: {count:,} ({count/total*100:.1f}%)")

Products with extracted features (feature): 876,231 / 1,134,566 (77.2%)

Feature coverage (feature column):
  Material: 554,113 (48.8%)
  Product_Type: 422,277 (37.2%)
  Features: 410,946 (36.2%)
  Dimensions: 379,742 (33.5%)
  Color: 224,110 (19.8%)
  Piece_Count: 111,638 (9.8%)
  Capacity_Volume: 74,241 (6.5%)
  Theme: 70,253 (6.2%)
  Size: 41,322 (3.6%)
  Brand: 41,297 (3.6%)
  Shape: 31,997 (2.8%)
  Sub_Type: 27,000 (2.4%)
  Shape_Style: 15,087 (1.3%)
  Thread_Count: 12,318 (1.1%)
  Pocket_Depth: 6,744 (0.6%)
  Scent: 5,799 (0.5%)
  Power_Rating: 3,791 (0.3%)
  Capacity: 2,400 (0.2%)
  Capacity_Cups: 1,522 (0.1%)
  Filter_Rating: 1,296 (0.1%)
  Voltage: 1,256 (0.1%)
  Part_Number: 413 (0.0%)
  Bar_Pressure: 359 (0.0%)
  Density_Weight: 262 (0.0%)
  Weight: 186 (0.0%)
  Stage_Count: 164 (0.0%)


In [51]:
# df_items_filtered.head(2)
# print(df_items_filtered.shape)
# print(df_items_filtered[df_items_filtered['extracted_features'].apply(len) == 0].shape)
df_items_filtered[df_items_filtered['extracted_features'].apply(len) != 0].head(2)

,category,tech1,description,title,tech2,brand,feature,rank,main_cat,price,asin,date,cat_1,cat_2,cat_3,cat_4,cat_5,cat_6,brand_clean,title_cleaned,extracted_features_title,description_cleaned,extracted_features_description,feature_cleaned,extracted_features_feature,extracted_features,all_text_cleaned,Bar_Pressure,Brand,Capacity,Capacity_Cups,Capacity_Volume,Color,Density_Weight,Dimensions,Features,Filter_Rating,Material,Part_Number,Piece_Count,Pocket_Depth,Power_Rating,Product_Type,Scent,Shape,Shape_Style,Size,Stage_Count,Sub_Type,Theme,Thread_Count,Voltage,Weight,capacity_numeric,capacity_unit,capacity_volume_numeric,capacity_volume_unit,piece_count_numeric,piece_count_unit,thread_count_numeric,thread_count_unit,weight_numeric,weight_unit,bar_pressure_numeric,capacity_cups_numeric,density_weight_lb,pocket_depth_in,power_rating_w,stage_count_numeric,voltage_numeric,dimension_1,dimension_2,dimension_3,dimension_unit
0,"['Home & Kitchen', 'Kitchen & Dining', 'Dining...",NaN,['It was a time honored tradition among the ea...,You Are Special Today Red Plate [With Red Pen],NaN,Waechtersbach USA,[],"['>#39,665 in Kitchen & Dining (See Top 100 in...",Amazon Home,$37.00,0001487795,"October 8, 2006",Home & Kitchen,Kitchen & Dining,Dining & Entertaining,Dinnerware,Plates,Dinner Plates,waechtersbachusa,special today red plate red pen,{'Color': 'red'},time honored tradition among early american fa...,{'Color': 'red'},,{},{'Color': 'red'},special today red plate red pen time honored t...,None,None,None,None,None,red,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"['Home & Kitchen', 'Kitchen & Dining', 'Dining...",NaN,"['16 oz squeeze bottle, 1 lb.']",Artistic Churchware Communion Cup Filler: RW525,NaN,Artistic Churchware,"['Religious Supply Center', 'RW-525', 'Communi...","['>#2,127,003 in Home & Kitchen (See Top 100 i...",Amazon Home,$12.48,0006564224,NaN,Home & Kitchen,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,Wine & Champagne Glasses,None,artisticchurchware,artistic churchware communion cup filler rw525,{'Product_Type': 'cup'},16 oz squeeze bottle 1 lb,{'Capacity_Volume': '16 oz'},religious supply center rw-525 communion cup f...,{'Product_Type': 'cup'},"{'Product_Type': 'cup', 'Capacity_Volume': '16...",artistic churchware communion cup filler rw525...,None,None,None,None,16 oz,None,None,None,None,None,None,None,None,None,None,cup,None,None,None,None,None,None,None,None,None,None,NaN,NaN,16.00,oz,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [52]:
df_items_filtered.shape

(1134566, 74)

## **Merge All Features Together**

In [25]:
# Combine title, description, and feature extracted features (title > description > feature priority)
def merge_features(row):
    """Merge extracted features from all three sources. Title takes highest priority."""
    title_feats = row['extracted_features_title']
    desc_feats = row['extracted_features_description']
    feat_feats = row['extracted_features_feature']
    # Start with lowest priority, overwrite with higher priority
    merged = {}
    if feat_feats:
        merged.update(feat_feats)
    if desc_feats:
        merged.update(desc_feats)
    if title_feats:
        merged.update(title_feats)
    return merged

df_items_filtered['extracted_features'] = df_items_filtered.apply(merge_features, axis=1)

# Combine cleaned text from all three sources
df_items_filtered['all_text_cleaned'] = (
    df_items_filtered['title_cleaned'].fillna('') + ' ' +
    df_items_filtered['description_cleaned'].fillna('') + ' ' +
    df_items_filtered['feature_cleaned'].fillna('')
).str.replace(r'\s+', ' ', regex=True).str.strip()

# Summary comparison
total = len(df_items_filtered)
title_count = (df_items_filtered['extracted_features_title'].apply(len) > 0).sum()
desc_count = (df_items_filtered['extracted_features_description'].apply(len) > 0).sum()
feat_count = (df_items_filtered['extracted_features_feature'].apply(len) > 0).sum()
combined_count = (df_items_filtered['extracted_features'].apply(len) > 0).sum()

print(f"Products with features (title only):       {title_count:,} / {total:,} ({title_count/total*100:.1f}%)")
print(f"Products with features (description only):  {desc_count:,} / {total:,} ({desc_count/total*100:.1f}%)")
print(f"Products with features (feature only):      {feat_count:,} / {total:,} ({feat_count/total*100:.1f}%)")
print(f"Products with features (combined):          {combined_count:,} / {total:,} ({combined_count/total*100:.1f}%)")

# Per-field comparison
field_counts_combined = Counter()
for feat_dict in df_items_filtered['extracted_features']:
    for key in feat_dict:
        field_counts_combined[key] += 1

print(f"\nFeature coverage comparison (title → combined):")
all_fields = set(field_counts_combined.keys())
for field in sorted(all_fields, key=lambda f: field_counts_combined[f], reverse=True):
    title_n = field_counts.get(field, 0)
    combined_n = field_counts_combined[field]
    diff = combined_n - title_n
    print(f"  {field}: {title_n:,} → {combined_n:,} (+{diff:,})")

Products with features (title only):       1,048,500 / 1,134,566 (92.4%)
Products with features (description only):  915,857 / 1,134,566 (80.7%)
Products with features (feature only):      876,231 / 1,134,566 (77.2%)
Products with features (combined):          1,112,733 / 1,134,566 (98.1%)

Feature coverage comparison (title → combined):
  Product_Type: 724,822 → 838,262 (+113,440)
  Material: 389,648 → 761,739 (+372,091)
  Features: 300,370 → 638,614 (+338,244)
  Dimensions: 302,258 → 591,890 (+289,632)
  Color: 351,924 → 530,245 (+178,321)
  Piece_Count: 188,953 → 251,125 (+62,172)
  Theme: 117,780 → 181,559 (+63,779)
  Brand: 144,362 → 152,854 (+8,492)
  Capacity_Volume: 71,510 → 124,755 (+53,245)
  Size: 76,125 → 81,941 (+5,816)
  Shape: 45,990 → 73,626 (+27,636)
  Sub_Type: 29,245 → 53,374 (+24,129)
  Shape_Style: 23,052 → 34,514 (+11,462)
  Thread_Count: 13,897 → 19,346 (+5,449)
  Scent: 8,155 → 14,411 (+6,256)
  Pocket_Depth: 2,852 → 9,499 (+6,647)
  Power_Rating: 2,033 → 6,070 

In [26]:
# Get the rows with missing extracted_features
print(f'Shape of the total dataset: {df_items.shape[0]}')
print(f'Shape of the filtered dataset: {df_items_filtered.shape[0]}')
print(f'Shape of items with no features extracted: {df_items_filtered[df_items_filtered['extracted_features'].apply(len) == 0].shape[0]}')

Shape of the total dataset: 1285392
Shape of the filtered dataset: 1134566
Shape of items with no features extracted: 21833


In [27]:
df_items_filtered.head(2)

,category,tech1,description,title,tech2,brand,feature,rank,main_cat,price,asin,date,cat_1,cat_2,cat_3,cat_4,cat_5,cat_6,brand_clean,title_cleaned,extracted_features_title,description_cleaned,extracted_features_description,feature_cleaned,extracted_features_feature,extracted_features,all_text_cleaned
0,"['Home & Kitchen', 'Kitchen & Dining', 'Dining...",NaN,['It was a time honored tradition among the ea...,You Are Special Today Red Plate [With Red Pen],NaN,Waechtersbach USA,[],"['>#39,665 in Kitchen & Dining (See Top 100 in...",Amazon Home,$37.00,0001487795,"October 8, 2006",Home & Kitchen,Kitchen & Dining,Dining & Entertaining,Dinnerware,Plates,Dinner Plates,waechtersbachusa,special today red plate red pen,{'Color': 'red'},time honored tradition among early american fa...,{'Color': 'red'},,{},{'Color': 'red'},special today red plate red pen time honored t...
1,"['Home & Kitchen', 'Home Dcor', 'Candles & Hol...",NaN,['VICKS INHALER relieves stuffy noses helps si...,Vicks Inhaler Relief for Cold Sinus Nasal Cong...,NaN,Vicks,[],"['>#1,763,185 in Home & Kitchen (See Top 100 i...",Amazon Home,$4.05,0002020300,NaN,Home & Kitchen,Home Dcor,Candles & Holders,Candles,None,None,vicks,vicks inhaler relief cold sinus nasal congesti...,{},vicks inhaler relief stuffy nose help sinus co...,{},,{},{},vicks inhaler relief cold sinus nasal congesti...


In [28]:
# Group by the categories by missing extracted_features (no items extracted for those products)
df_items_counts = df_items_filtered[df_items_filtered['extracted_features'].apply(len) == 0] \
    .groupby('cat_3') \
    .size() \
    .reset_index(name='count') \
    .sort_values('count', ascending=False)

df_items_counts[df_items_counts['count'] > 1000]

,cat_3,count
55,Storage & Organization,2420
28,Home Dcor Accents,2302
38,Kitchen Utensils & Gadgets,2160
54,Small Appliances,1707
48,Posters & Prints,1672
21,Dining & Entertaining,1200


In [29]:
# Filter to items with cat_3 in master_metadata and expand extracted_features into columns

# Get all unique keys from extracted_features
all_keys = set()
for feat_dict in df_items_filtered['extracted_features']:
    all_keys.update(feat_dict.keys())

# Create a column for each key
for key in sorted(all_keys):
    df_items_filtered[key] = df_items_filtered['extracted_features'].apply(lambda x: x.get(key))

print(f"Rows: {len(df_items_filtered):,}")
print(f"Feature columns created: {sorted(all_keys)}")
print(f"\nMissing values per feature:")
for key in sorted(all_keys):
    missing = df_items_filtered[key].isna().sum()
    print(f"  {key}: {missing:,} ({missing/len(df_items_filtered)*100:.1f}% missing)")

Rows: 1,134,566
Feature columns created: ['Bar_Pressure', 'Brand', 'Capacity', 'Capacity_Cups', 'Capacity_Volume', 'Color', 'Density_Weight', 'Dimensions', 'Features', 'Filter_Rating', 'Material', 'Part_Number', 'Piece_Count', 'Pocket_Depth', 'Power_Rating', 'Product_Type', 'Scent', 'Shape', 'Shape_Style', 'Size', 'Stage_Count', 'Sub_Type', 'Theme', 'Thread_Count', 'Voltage', 'Weight']

Missing values per feature:
  Bar_Pressure: 1,134,106 (100.0% missing)
  Brand: 981,712 (86.5% missing)
  Capacity: 1,130,540 (99.6% missing)
  Capacity_Cups: 1,131,865 (99.8% missing)
  Capacity_Volume: 1,009,811 (89.0% missing)
  Color: 604,321 (53.3% missing)
  Density_Weight: 1,134,225 (100.0% missing)
  Dimensions: 542,676 (47.8% missing)
  Features: 495,952 (43.7% missing)
  Filter_Rating: 1,132,430 (99.8% missing)
  Material: 372,827 (32.9% missing)
  Part_Number: 1,133,959 (99.9% missing)
  Piece_Count: 883,441 (77.9% missing)
  Pocket_Depth: 1,125,067 (99.2% missing)
  Power_Rating: 1,128,496 (

In [30]:
import re
import numpy as np

def parse_numeric_and_unit(value):
    """Parse a string like '12 oz' or '16-oz' into (numeric, unit)."""
    if pd.isna(value) or not isinstance(value, str) or value.strip() == '':
        return np.nan, np.nan
    value = value.strip().lower().replace('-', ' ')
    match = re.match(r'^([\d.]+)\s*([a-z%]+.*)?$', value)
    if match:
        num = float(match.group(1))
        unit = match.group(2).strip() if match.group(2) else np.nan
        return num, unit
    return np.nan, np.nan

# Fields to parse into numeric + unit (2 columns each)
numeric_unit_fields = [
    'Capacity', 'Capacity_Volume', 'Piece_Count',
    'Thread_Count', 'Weight'
]

for field in numeric_unit_fields:
    col_numeric = field.lower() + '_numeric'
    col_unit = field.lower() + '_unit'
    df_items_filtered[[col_numeric, col_unit]] = df_items_filtered[field].apply(
        lambda x: pd.Series(parse_numeric_and_unit(x))
    )

# Fields with only 1 unique unit: extract numeric only, name includes the unit
numeric_only_fields = {
    'Bar_Pressure': 'bar_pressure_numeric',
    'Capacity_Cups': 'capacity_cups_numeric',
    'Density_Weight': 'density_weight_lb',
    'Pocket_Depth': 'pocket_depth_in',
    'Power_Rating': 'power_rating_w',
    'Stage_Count': 'stage_count_numeric',
    'Voltage': 'voltage_numeric',
}

for field, col_name in numeric_only_fields.items():
    df_items_filtered[col_name] = df_items_filtered[field].apply(
        lambda x: parse_numeric_and_unit(x)[0]
    )

def parse_dimensions(value):
    """Parse dimension string into (dim1, dim2, dim3, unit).
    Handles formats: '25x20x10 in', '12 in x 18 in', '12in x 18in', '32x-30', etc.
    dim1 >= dim2 >= dim3. If only 2 dimensions, dim3 is NaN."""
    if pd.isna(value) or not isinstance(value, str) or value.strip() == '':
        return np.nan, np.nan, np.nan, np.nan
    value = value.strip().lower()
    # Pattern: number [unit] x[-] number [unit] [x[-] number [unit]]
    match = re.match(
        r'^([\d.]+)\s*([a-z]*)\s*[x×]\s*-?\s*([\d.]+)\s*([a-z]*)\s*(?:[x×]\s*-?\s*([\d.]+)\s*([a-z]*))?$',
        value
    )
    if match:
        dims = [float(match.group(1)), float(match.group(3))]
        if match.group(5):
            dims.append(float(match.group(5)))
        dims.sort(reverse=True)
        # Get unit from the last available unit group
        unit = match.group(6) or match.group(4) or match.group(2) or np.nan
        unit = unit.strip() if isinstance(unit, str) and unit.strip() else np.nan
        dim1 = dims[0]
        dim2 = dims[1] if len(dims) >= 2 else np.nan
        dim3 = dims[2] if len(dims) >= 3 else np.nan
        return dim1, dim2, dim3, unit
    # Single dimension like "25 in"
    match = re.match(r'^([\d.]+)\s*([a-z]+.*)?$', value)
    if match:
        dim1 = float(match.group(1))
        unit = match.group(2).strip() if match.group(2) else np.nan
        return dim1, np.nan, np.nan, unit
    return np.nan, np.nan, np.nan, np.nan

# Parse Dimensions into dim1, dim2, dim3, unit
df_items_filtered[['dimension_1', 'dimension_2', 'dimension_3', 'dimension_unit']] = df_items_filtered['Dimensions'].apply(
    lambda x: pd.Series(parse_dimensions(x))
)

# Standardize unit names using unit_map on all parsed unit columns
all_unit_cols = [f.lower() + '_unit' for f in numeric_unit_fields] + ['dimension_unit']
for col in all_unit_cols:
    df_items_filtered[col] = df_items_filtered[col].apply(
        lambda x: unit_map.get(x, x) if isinstance(x, str) else x
    )

# Summary
print("New variables created and units standardized:\n")
print("--- Fields with numeric + unit columns ---")
for field in numeric_unit_fields:
    col_numeric = field.lower() + '_numeric'
    col_unit = field.lower() + '_unit'
    n = df_items_filtered[col_numeric].notna().sum()
    units = df_items_filtered[col_unit].dropna().unique()
    print(f"{field}: {n:,} non-null | units: {sorted(units)[:10]}")

print("\n--- Fields with numeric only (single unit) ---")
for field, col_name in numeric_only_fields.items():
    n = df_items_filtered[col_name].notna().sum()
    print(f"{field} → {col_name}: {n:,} non-null")

n1 = df_items_filtered['dimension_1'].notna().sum()
n2 = df_items_filtered['dimension_2'].notna().sum()
n3 = df_items_filtered['dimension_3'].notna().sum()
d_units = df_items_filtered['dimension_unit'].dropna().unique()
print(f"\nDimensions: dim1={n1:,}, dim2={n2:,}, dim3={n3:,} non-null | units: {sorted(d_units)[:10]}")

New variables created and units standardized:

--- Fields with numeric + unit columns ---
Capacity: 4,026 non-null | units: ['cup', 'l', 'oz']
Capacity_Volume: 124,755 non-null | units: ['bottle', 'cubic foot', 'cup', 'fl oz', 'g', 'gal', 'l', 'lb', 'ml', 'oz']
Piece_Count: 157,366 non-null | units: ['bottle', 'capacity', 'chair', 'cone', 'count', 'door', 'drawer', 'dz', 'hook', 'in 1']
Thread_Count: 19,346 non-null | units: ['count', 'series', 'thread count']
Weight: 252 non-null | units: ['g', 'lb']

--- Fields with numeric only (single unit) ---
Bar_Pressure → bar_pressure_numeric: 460 non-null
Capacity_Cups → capacity_cups_numeric: 2,701 non-null
Density_Weight → density_weight_lb: 341 non-null
Pocket_Depth → pocket_depth_in: 8,507 non-null
Power_Rating → power_rating_w: 6,070 non-null
Stage_Count → stage_count_numeric: 362 non-null
Voltage → voltage_numeric: 2,363 non-null

Dimensions: dim1=583,817, dim2=332,516, dim3=39,759 non-null | units: ['cm', 'cm - m', 'cm --m', 'cm cm', 'c

## Feature Cleaning

In [ ]:
# Copy df_items_filtered to preserve original values, then filter numeric features to valid ranges
df_items_filtered_cleaned = df_items_filtered.copy()

valid_ranges = {
    'bar_pressure_numeric': (1, 25),
    'capacity_cups_numeric': (1, 30),
    'density_weight_lb': (0.5, 30),
    'pocket_depth_in': (5, 25),
    'power_rating_w': (1, 5000),
    'stage_count_numeric': (1, 10),
    'voltage_numeric': (110, 240),
    'thread_count_numeric': (80, 2000),
    'weight_numeric': (0.1, 300),
}

for col, (low, high) in valid_ranges.items():
    before = df_items_filtered_cleaned[col].notna().sum()
    df_items_filtered_cleaned.loc[~df_items_filtered_cleaned[col].between(low, high), col] = np.nan
    after = df_items_filtered_cleaned[col].notna().sum()
    removed = before - after
    print(f"{col} [{low}-{high}]: {before:,} → {after:,} ({removed:,} removed, {removed/before*100:.1f}% of non-null)" if before > 0 else f"{col}: no data")

## SBERT Embeddings

In [32]:
# Download SBERT and create the title_embedding variable
from sentence_transformers import SentenceTransformer

# Load SBERT model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Get non-empty title_cleaned values
titles = df_items_filtered['title_cleaned'].fillna('').tolist()

# Encode in batches
print(f"Encoding {len(titles):,} titles...")
title_embeddings = model.encode(titles, batch_size=256, show_progress_bar=True)

# Store embeddings
df_items_filtered['title_embedding'] = list(title_embeddings)

print(f"\nEmbedding shape: {title_embeddings.shape}")
print(f"Sample embedding (first 10 dims): {title_embeddings[0][:10]}")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10851.06it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding 1,134,566 titles...


Batches:  10%|█         | 450/4432 [01:30<13:19,  4.98it/s]


KeyboardInterrupt: 

## Save the Results

In [ ]:
# Save df_items_filtered and df_items for analysis notebook
# Drop the embedding column for CSV (store separately)
import pickle

# Save full dataframe with embeddings as pickle
df_items_filtered.to_pickle('../data/df_items_filtered.pkl')
df_items.to_pickle('../data/df_items.pkl')
df_combined.to_pickle('../data/df_combined.pkl')

print(f'Saved df_items_filtered: {df_items_filtered.shape}')
print(f'Saved df_items: {df_items.shape}')
print(f'Saved df_combined: {df_combined.shape}')